# Análise Exploratória de Dados — Telco Customer Churn

## Contexto do Problema de Negócio

Uma empresa de telecomunicações enfrenta alta taxa de **churn** (cancelamento de clientes). Adquirir um novo cliente custa de 5 a 25 vezes mais do que reter um existente, tornando a retenção proativa uma alavanca de negócio crítica.

### Objetivo
Construir um modelo preditivo que identifique clientes com risco de cancelar o serviço, permitindo ações preventivas pela equipe de retenção.

### Stakeholders
- **Equipe de Retenção/CRM** — principal consumidor do modelo
- **Gestão de Produto** — usa insights para priorizar melhorias no serviço
- **Área Financeira** — avalia ROI das ações de retenção
- **Equipe de Dados/ML** — constrói e mantém o modelo

### Métricas de Sucesso
**Técnicas:** ROC-AUC, PR-AUC, F1-Score, Precision, Recall

**KPI de negócio:** Redução da taxa de churn mensal em pelo menos 15% após ações preventivas baseadas no modelo, medida por comparação com taxa histórica ou grupo de controle.

### Trade-off Central
- **Falso Negativo** → cliente cancela sem ser detectado → perda de receita
- **Falso Positivo** → campanha de retenção desnecessária → custo operacional

### Objetivo
Entender a estrutura, qualidade e padrões dos dados antes de construir qualquer modelo. Identificar variáveis relevantes, problemas de qualidade e hipóteses iniciais.

## 1. Configuração do Ambiente

In [ ]:
# Bibliotecas basicas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Visualizacao
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Estatistica
from scipy import stats

# Configuracoes de exibicao
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

## 2. Carregamento dos Dados

### Sobre o Dataset Telco Customer Churn

Fonte: IBM Sample Dataset — 7.043 clientes de uma empresa de telecom fictícia, 33 colunas.

**Grupos de variáveis:**
- **Identificação:** `CustomerID`
- **Demográficas:** `Gender`, `Senior Citizen`, `Partner`, `Dependents`
- **Contrato:** `Tenure Months`, `Contract`, `Paperless Billing`, `Payment Method`
- **Serviços:** `Phone Service`, `Multiple Lines`, `Internet Service`, `Online Security`, `Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies`
- **Financeiro:** `Monthly Charges`, `Total Charges`, `CLTV`
- **Target:** `Churn Label` (Yes/No), `Churn Value` (1/0), `Churn Score`, `Churn Reason`

In [ ]:
PATH = "../data/raw/Telco_customer_churn.xlsx"

df = pd.read_excel(PATH)

print(f"Shape dos dados: {df.shape}")
print(f"\nPrimeiras linhas do dataset:")
df.head(10)

In [ ]:
print("=== INFORMACOES GERAIS DO DATASET ===\n")
print(df.info())

print("\n=== ESTATISTICAS DESCRITIVAS (numericas) ===\n")
df.describe()

> **O que observar:**
> - `df.info()`: tipos de dados, contagem de não-nulos e uso de memória.
> - `Total Charges` aparece como `object` apesar de ser monetário — há valores inválidos (espaços em branco).
> - `df.describe()`: escala, dispersão e possíveis outliers nas variáveis numéricas.

## 3. Análise Exploratória de Dados (EDA)

### 3.1 Análise de Missing Values

**Objetivo:** Identificar valores ausentes e entender seu padrão antes de qualquer transformação.

In [ ]:
print("=== ANALISE DE MISSING VALUES ===\n")

missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Pct": (df.isnull().sum() / len(df) * 100).round(2)
}).query("Missing_Count > 0").sort_values("Missing_Pct", ascending=False)

if len(missing) > 0:
    print(missing)
    plt.figure(figsize=(8, 4))
    plt.barh(missing.index, missing["Missing_Pct"], color="coral")
    plt.xlabel("Porcentagem de Missing Values (%)")
    plt.title("Distribuicao de Missing Values por Coluna")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum missing value detectado!")

# Investigar Total Charges
print("\n=== INVESTIGACAO: Total Charges ===")
mask_invalido = df["Total Charges"].astype(str).str.strip() == ""
print(f"Registros com Total Charges em branco: {mask_invalido.sum()}")
print(df[mask_invalido][["CustomerID", "Tenure Months", "Monthly Charges", "Total Charges", "Churn Label"]])

> **Interpretação:**
> - `Churn Reason` tem ~73% de nulos — **esperado**: só clientes que cancelaram têm motivo registrado. Não é problema de qualidade.
> - `Total Charges` com espaço em branco ocorre em clientes com `Tenure Months = 0` (ainda não receberam fatura). O valor correto é `0`.

### 3.2 Análise da Variável Target

**Objetivo:** Entender a distribuição da variável alvo e quantificar o desbalanceamento de classes.

In [ ]:
print("=== DISTRIBUICAO DA VARIAVEL TARGET ===\n")

target_counts = df["Churn Label"].value_counts()
target_pct = df["Churn Label"].value_counts(normalize=True) * 100

for label, pct in target_pct.items():
    print(f"{label}: {target_counts[label]:,} clientes ({pct:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(target_counts.index, target_counts.values, color=["steelblue", "coral"])
for i, (label, val) in enumerate(target_counts.items()):
    axes[0].text(i, val + 30, f"{val:,}\n({target_pct[label]:.1f}%)", ha="center", fontweight="bold")
axes[0].set_ylabel("Frequencia")
axes[0].set_title("Distribuicao do Churn")
axes[0].grid(axis="y", alpha=0.3)

axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct="%1.1f%%", colors=["steelblue", "coral"], startangle=90)
axes[1].set_title("Proporcao de Classes")

plt.suptitle("Variavel Target: Churn Label", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

ratio = target_counts.min() / target_counts.max()
print(f"\nRatio de balanceamento: {ratio:.2f}")
print("Desbalanceamento moderado (1:2.8) — considerar class_weight ou threshold tuning.")

### 3.3 Análise de Distribuições — Variáveis Numéricas

**Objetivo:** Entender a distribuição de `Tenure Months`, `Monthly Charges` e `Total Charges`, identificar assimetrias e separar por Churn.

**Também convertemos `Total Charges` de `object` para `float` aqui.**

In [ ]:
# Corrigir Total Charges: espacos em branco -> 0
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)

numeric_cols = ["Tenure Months", "Monthly Charges", "Total Charges"]

print("=== ASSIMETRIA (SKEWNESS) E CURTOSE ===\n")
dist_stats = []
for col in numeric_cols:
    skew = df[col].skew()
    kurt = df[col].kurtosis()
    interp = "Simetrica" if abs(skew) < 0.5 else ("Assimetrica direita" if skew > 0 else "Assimetrica esquerda")
    dist_stats.append({"Coluna": col, "Skewness": round(skew, 3), "Kurtosis": round(kurt, 3), "Interpretacao": interp})

print(pd.DataFrame(dist_stats).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, col in enumerate(numeric_cols):
    ax = axes[0, idx]
    df[col].hist(bins=40, ax=ax, color="skyblue", edgecolor="black", alpha=0.7)
    ax.set_title(f"{col}\nSkew: {df[col].skew():.2f}", fontweight="bold")
    ax.set_xlabel("Valor"); ax.set_ylabel("Frequencia")
    ax.grid(axis="y", alpha=0.3)

    ax2 = axes[1, idx]
    for label, color in [("No", "steelblue"), ("Yes", "coral")]:
        subset = df[df["Churn Label"] == label][col]
        ax2.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor="white")
    ax2.set_title(f"{col} por Churn", fontweight="bold")
    ax2.set_xlabel("Valor"); ax2.set_ylabel("Frequencia")
    ax2.legend(title="Churn")
    ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Distribuicoes das Variaveis Numericas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

> **O que observar:**
> - `Tenure Months`: distribuição **bimodal** — pico em clientes novos (0-12m) e antigos (>60m). Clientes com churn se concentram nos primeiros meses.
> - `Monthly Charges`: clientes com churn têm mensalidade mais alta em média.
> - `Total Charges`: fortemente correlacionado com `Tenure × Monthly Charges` — verificaremos multicolinearidade na seção 3.6.

### 3.4 Análise de Outliers

**Objetivo:** Identificar valores extremos com boxplots e Z-Score (desvios > 3σ).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[idx], color="coral")
    axes[idx].set_title(f"Boxplot: {col}", fontweight="bold")
    axes[idx].set_xlabel(col)
    axes[idx].grid(axis="x", alpha=0.3)

    z_scores = np.abs(stats.zscore(df[col].dropna()))
    outlier_count = (z_scores > 3).sum()
    axes[idx].text(0.98, 0.98, f"Outliers (Z>3): {outlier_count}",
                   transform=axes[idx].transAxes, fontsize=9,
                   va="top", ha="right",
                   bbox=dict(facecolor="white", alpha=0.6, edgecolor="gray"))

plt.suptitle("Analise de Outliers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("=== DESCRICAO ESTATISTICA DOS OUTLIERS ===\n")
for col in numeric_cols:
    z = np.abs(stats.zscore(df[col].dropna()))
    n = (z > 3).sum()
    pct = n / len(df) * 100
    print(f"{col}: {n} outliers ({pct:.1f}%)  |  max={df[col].max():.2f}  |  P99={df[col].quantile(0.99):.2f}")

> **Conclusão:** Outliers inexpressivos (<1%). Não há necessidade de remoção — serão tratados com normalização no pré-processamento.

### 3.5 Análise de Anomalias por Conhecimento de Domínio

**Objetivo:** Verificar consistências usando regras de negócio do setor de telecom.

In [ ]:
print("=== ANOMALIAS POR CONHECIMENTO DE DOMINIO ===\n")

anomalies = []

# 1. Tenure = 0 com Total Charges > 0
a1 = df[(df["Tenure Months"] == 0) & (df["Total Charges"] > 0)]
if len(a1):
    anomalies.append(f"Tenure=0 mas Total Charges>0: {len(a1)} casos")

# 2. Clientes sem nenhum servico
a2 = df[(df["Phone Service"] == "No") & (df["Internet Service"] == "No Internet Service")]
print(f"Clientes sem phone nem internet: {len(a2)}")

# 3. Servicos de internet contratados sem Internet Service
internet_services = ["Online Security", "Online Backup", "Device Protection",
                     "Tech Support", "Streaming TV", "Streaming Movies"]
mask_no_internet = df["Internet Service"] == "No Internet Service"
for svc in internet_services:
    inconsistente = df[mask_no_internet & (df[svc] == "Yes")]
    if len(inconsistente):
        anomalies.append(f"{svc}=Yes mas Internet Service=No: {len(inconsistente)} casos")

# 4. Duplicados
dups = df.duplicated().sum()
print(f"Registros duplicados: {dups}")

if anomalies:
    print("\nAnomalias detectadas:")
    for a in anomalies:
        print(f"  - {a}")
else:
    print("\nNenhuma anomalia de dominio detectada. Dados consistentes.")

### 3.6 Análise das Variáveis Categóricas

**Objetivo:** Entender como cada variável categórica se relaciona com o churn.

**Bloco 1 — Variáveis demográficas**

In [ ]:
cat_cols_demo = ["Gender", "Senior Citizen", "Partner", "Dependents"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(cat_cols_demo):
    churn_rate = df.groupby(col)["Churn Value"].mean().sort_values(ascending=False)
    ax = axes[idx]
    colors = ["coral" if i == 0 else "steelblue" for i in range(len(churn_rate))]
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values * 100, color=colors)
    ax.set_title(f"Churn Rate por {col}", fontweight="bold")
    ax.set_ylabel("Churn Rate (%)")
    ax.set_ylim(0, 60)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val*100:.1f}%", ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Churn Rate por Variaveis Demograficas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Bloco 2 — Variaveis de contrato
cat_cols_contract = ["Contract", "Payment Method", "Paperless Billing", "Internet Service"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(cat_cols_contract):
    churn_rate = df.groupby(col)["Churn Value"].mean().sort_values(ascending=False)
    ax = axes[idx]
    colors = ["coral" if i == 0 else "steelblue" for i in range(len(churn_rate))]
    bars = ax.barh(churn_rate.index.astype(str), churn_rate.values * 100, color=colors)
    ax.set_title(f"Churn Rate por {col}", fontweight="bold")
    ax.set_xlabel("Churn Rate (%)")
    ax.set_xlim(0, 80)
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val*100:.1f}%", va="center", fontsize=10, fontweight="bold")

plt.suptitle("Churn Rate por Variaveis de Contrato", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Bloco 3 — Servicos contratados
cat_cols_services = [
    "Phone Service", "Multiple Lines", "Online Security", "Online Backup",
    "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for idx, col in enumerate(cat_cols_services):
    churn_rate = df.groupby(col)["Churn Value"].mean().sort_values(ascending=False)
    counts = df[col].value_counts()
    ax = axes[idx]
    colors = ["coral" if i == 0 else "steelblue" for i in range(len(churn_rate))]
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values * 100, color=colors)
    ax.set_title(f"{col}", fontweight="bold", fontsize=10)
    ax.set_ylabel("Churn Rate (%)")
    ax.set_ylim(0, 70)
    ax.tick_params(axis="x", rotation=15, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    for bar, (cat, val) in zip(bars, churn_rate.items()):
        n = counts.get(cat, 0)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val*100:.0f}%\n(n={n:,})", ha="center", fontsize=8)

plt.suptitle("Churn Rate por Servicos Contratados", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

> **Hipóteses confirmadas:**
> - **Contrato `Month-to-month`:** ~42% de churn vs. 11% (`One year`) e 3% (`Two year`) — variável mais preditiva do dataset.
> - **Sem `Tech Support` / `Online Security`:** churn ~2x maior que clientes com esses serviços.
> - **`Electronic check`:** ~45% de churn — possível correlação com menor engajamento/fidelização.
> - **`Senior Citizen`:** 41% vs. 24% — segmento vulnerável que merece atenção especial da equipe de retenção.

### 3.7 Análise de Correlações

**Objetivo:** Identificar relações entre variáveis numéricas, detectar multicolinearidade e **data leakage**.

In [ ]:
df_corr = df[["Tenure Months", "Monthly Charges", "Total Charges",
              "Churn Score", "CLTV", "Churn Value"]].copy()

corr_matrix = df_corr.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title("Matriz de Correlacao — Variaveis Numericas", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

print("=== CORRELACOES COM O TARGET (Churn Value) ===\n")
print(corr_matrix["Churn Value"].sort_values(ascending=False))

print("\n=== MULTICOLINEARIDADE (correlacao > 0.7 entre features) ===\n")
feats = ["Tenure Months", "Monthly Charges", "Total Charges", "Churn Score", "CLTV"]
high = []
for i in range(len(feats)):
    for j in range(i+1, len(feats)):
        val = corr_matrix.loc[feats[i], feats[j]]
        if abs(val) > 0.7:
            high.append({"Feature 1": feats[i], "Feature 2": feats[j], "Correlacao": round(val, 3)})

if high:
    print(pd.DataFrame(high).to_string(index=False))
    print("\nAtencao: alta correlacao pode indicar multicolinearidade ou data leakage.")
else:
    print("Nenhuma correlacao forte detectada entre features.")

> **Atenção — Data Leakage:**
> - `Churn Score` e `CLTV` são scores **calculados pela empresa após o evento de churn** — não devem entrar no modelo preditivo.
> - `Total Charges` é fortemente correlacionado com `Tenure × Monthly Charges`. Decisão de manter ou descartar será feita no feature engineering.

### 3.8 Análise de Churn Reason

**Objetivo:** Entender *por que* os clientes cancelam — insumo estratégico para os stakeholders de negócio.

In [ ]:
df_churned = df[df["Churn Label"] == "Yes"].copy()

reason_counts = df_churned["Churn Reason"].value_counts()

plt.figure(figsize=(12, 7))
colors = ["coral"] * 5 + ["steelblue"] * (len(reason_counts) - 5)
reason_counts.plot(kind="barh", color=colors[::-1])
plt.xlabel("Numero de Clientes")
plt.title("Motivos de Cancelamento", fontsize=14, fontweight="bold")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
for i, (reason, val) in enumerate(reason_counts.items()):
    plt.text(val + 1, i, f"{val} ({val/len(df_churned)*100:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("=== AGRUPAMENTO POR CATEGORIA DE MOTIVO ===\n")
categories = {
    "Competitor":      ["Competitor made better offer", "Competitor had better devices",
                        "Competitor offered more data", "Competitor offered higher download speeds"],
    "Service/Support": ["Attitude of support person", "Attitude of service provider",
                        "Poor expertise of phone support", "Poor expertise of online support",
                        "Network reliability", "Service dissatisfaction"],
    "Product":         ["Product dissatisfaction", "Lack of self-service on Website",
                        "Limited range of services", "Lack of affordable download/upload speed"],
    "Price":           ["Price too high", "Extra data charges", "Long distance charges"],
    "Other":           ["Moved", "Deceased", "Don't know"]
}
cat_totals = {cat: df_churned["Churn Reason"].isin(reasons).sum() for cat, reasons in categories.items()}
for cat, total in sorted(cat_totals.items(), key=lambda x: -x[1]):
    print(f"{cat:20s}: {total:4d} clientes ({total/len(df_churned)*100:.1f}%)")

> **Insight para stakeholders:**
> - **~45% dos cancelamentos são por concorrência** — o modelo identifica clientes em risco, mas a retenção exige ações de pricing e produto.
> - **~25% são por atendimento/suporte** — melhorias operacionais têm impacto direto.
> - Esta análise é **pós-evento** e não entra no modelo — serve para informar o time de CRM sobre o contexto dos cancelamentos.

## 4. Sumário da EDA e Decisões para Modelagem

### 4.1 Colunas a Descartar

In [ ]:
COLUNAS_DESCARTAR = {
    "Identificadores": ["CustomerID", "Count"],
    "Geolocalizacao (sem variancia util)": ["Country", "State", "City", "Zip Code", "Lat Long", "Latitude", "Longitude"],
    "Data Leakage (derivados do target)": ["Churn Score", "CLTV"],
    "Target / pos-evento": ["Churn Label", "Churn Reason"],
}

print("=== COLUNAS A DESCARTAR DO MODELO ===\n")
total_desc = 0
for motivo, cols in COLUNAS_DESCARTAR.items():
    print(f"{motivo}:")
    for c in cols:
        print(f"  - {c}")
    total_desc += len(cols)

colunas_modelo = [c for c in df.columns
                  if c not in [col for cols in COLUNAS_DESCARTAR.values() for col in cols]
                  and c != "Churn Value"]

print(f"\nTotal descartadas: {total_desc}")
print(f"Features para o modelo ({len(colunas_modelo)}): {colunas_modelo}")

### 4.2 Hipóteses Confirmadas pela EDA

| # | Hipótese | Evidência |
|---|---|---|
| 1 | Contrato mensal é o maior preditor de churn | Churn rate 42% vs 3% (two-year) |
| 2 | Clientes novos (tenure < 12 meses) têm maior risco | Pico de churn nos primeiros meses |
| 3 | Monthly Charges alto eleva o risco | Distribuição deslocada nos clientes que cancelaram |
| 4 | Ausência de serviços de proteção eleva o risco | Churn ~2x maior sem Security/TechSupport |
| 5 | Idosos e clientes sem parceiro têm mais churn | Senior Citizen: 41% vs 24% |
| 6 | Pagamento por electronic check correlaciona com churn | Churn rate ~45% |

### 4.3 Próximos Passos (Estágio 2)

1. **Feature Engineering:** encoding das categóricas, binning de `Tenure Months`, criação de `avg_monthly_spend`
2. **Baseline:** Dummy Classifier + Regressão Logística com log no MLflow
3. **MLP em PyTorch** com arquitetura configurável e early stopping
4. **Análise de custo FP vs FN** para definição do threshold de decisão

In [ ]:
# Salvar dataset com correcao de Total Charges para uso nas proximas etapas
df.to_csv("../data/processed/telco_churn_clean.csv", index=False)
print("Dataset salvo em: data/processed/telco_churn_clean.csv")
print(f"Shape: {df.shape}")